# Final Project Notebook: RAG Learning Assistant over Course PDFs

This notebook documents and validates the final technical pipeline for the project. It is designed to run end-to-end without exposing API keys, without forcing expensive OCR, and without rebuilding the FAISS index when an existing index is available.

The project satisfies the final technical requirements:

- FastAPI deployment is provided through `api.py`.
- Streamlit front-end is provided through `app.py`.
- The RAG pipeline uses FAISS over course PDFs.
- Transformer-based models are used for embeddings and generation.
- Retrieval evaluation reports Hit@3, Hit@5, and MRR.
- The notebook loads existing artifacts by default and only rebuilds the knowledge base if the index is missing.

## 1. Project Overview

The system is an AI-powered learning assistant for course materials. It processes PDFs, extracts text with OCR fallback when needed, chunks the text, embeds chunks with a SentenceTransformer model, stores vectors in FAISS, retrieves relevant context for a query, and generates study outputs such as answers, summaries, and MCQs.

The main user interface is Streamlit, while FastAPI provides a backend API for deployment or integration.

## 2. Environment and Imports

This cell imports the project modules needed for validation. It does not print environment variables or API keys.

In [ ]:
import json
import os
import sys
import time
from pathlib import Path

import pandas as pd

# Avoid network checks during notebook execution; models should already be available locally.
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.pipeline import build_knowledge_base
from src.retriever import Retriever
from src.vector_db import index_exists, load_index
from src.qa_generator import QAGenerator
from src.summarizer import Summarizer
from src.mcq_generator import MCQGenerator
from src.evaluator import run_full_evaluation

print(f"Project root: {ROOT}")
print("Core imports: OK")

## 3. Configuration and Paths

The notebook uses project-relative paths. It checks for the dataset, diagnostics, evaluation files, and FAISS index artifacts. It does not read or print `.env` contents.

In [ ]:
DATASET_DIR = ROOT / "dataset"
DATA_DIR = ROOT / "data"
DIAGNOSTICS_DIR = DATA_DIR / "diagnostics"
EVALUATION_DIR = DATA_DIR / "evaluation"
VECTOR_DIR = ROOT / "vector_store" / "faiss_index"

paths = {
    "dataset_dir": DATASET_DIR,
    "diagnostics_dir": DIAGNOSTICS_DIR,
    "evaluation_dir": EVALUATION_DIR,
    "faiss_index": VECTOR_DIR / "index.faiss",
    "faiss_metadata": VECTOR_DIR / "metadata.json",
    "eval_questions_revised": EVALUATION_DIR / "eval_questions_revised.csv",
}

pd.DataFrame(
    [{"name": k, "path": str(v), "exists": v.exists()} for k, v in paths.items()]
)

## 4. Dataset Inspection

Known final dataset facts:

- 6 PDFs total.
- 6/6 PDFs usable after pypdf extraction plus OCR fallback.
- OCR uses Poppler and Tesseract.
- OCR was needed for the KAN and MLOps PDFs.
- The final index contains 137 chunks from 6 sources.

The cell below loads existing diagnostics if available. It does not run full OCR.

In [ ]:
def load_json_if_exists(path: Path):
    if path.exists():
        with path.open("r", encoding="utf-8") as f:
            return json.load(f)
    return None

DATASET_SUMMARY_PATH = DIAGNOSTICS_DIR / "dataset_inspection_summary.json"
CHUNK_SUMMARY_PATH = DIAGNOSTICS_DIR / "chunk_statistics_summary.json"
BENCHMARK_PATH = DIAGNOSTICS_DIR / "final_benchmark_report.json"

dataset_summary = load_json_if_exists(DATASET_SUMMARY_PATH)
chunk_summary = load_json_if_exists(CHUNK_SUMMARY_PATH)
benchmark_report = load_json_if_exists(BENCHMARK_PATH)

print("Dataset diagnostics found:", dataset_summary is not None)
print("Chunk diagnostics found:", chunk_summary is not None)
print("Benchmark report found:", benchmark_report is not None)

dataset_files = sorted(DATASET_DIR.glob("*.pdf")) if DATASET_DIR.exists() else []
pd.DataFrame([{"pdf": p.name, "size_mb": round(p.stat().st_size / 1_000_000, 2)} for p in dataset_files])

## 5. PDF Processing and OCR Explanation

The ingestion pipeline processes PDFs page by page:

1. pypdf attempts normal text extraction.
2. Pages with insufficient extracted text use OCR fallback.
3. Poppler renders PDF pages into images.
4. Tesseract extracts text from those rendered images.
5. Page-level documents are chunked and embedded.

This notebook does not force OCR during normal execution. It loads existing processed files and the FAISS index when available.

In [ ]:
if dataset_summary:
    print(json.dumps(dataset_summary, indent=2, ensure_ascii=False)[:2000])
else:
    print("No dataset inspection summary found. Known final fact: 6/6 PDFs usable after pypdf + OCR.")

## 6. Build or Load Knowledge Base

If the FAISS index already exists, this notebook loads it. If it does not exist, the notebook runs the full knowledge-base build once. That fallback may perform PDF processing and OCR, so it is only triggered when the index is missing.

In [ ]:
if index_exists():
    index, metadata = load_index()
    kb_status = {
        "status": "loaded_existing_index",
        "num_vectors": int(index.ntotal),
        "num_metadata_rows": len(metadata),
        "num_sources": len({m["source"] for m in metadata}),
    }
else:
    print("FAISS index missing. Building knowledge base. This may run PDF processing/OCR.")
    kb_status = build_knowledge_base()
    index, metadata = load_index()

kb_status

## 7. Chunk Statistics

The final knowledge base contains 137 chunks from 6 sources. The cell below reads the saved chunk diagnostics if available and also summarizes loaded FAISS metadata.

In [ ]:
source_counts = pd.Series([m["source"] for m in metadata]).value_counts().rename_axis("source").reset_index(name="chunks")
print("Known final chunks: 137")
print("Loaded metadata chunks:", len(metadata))
print("Loaded sources:", source_counts.shape[0])

if chunk_summary:
    print("Chunk summary JSON:")
    print(json.dumps(chunk_summary, indent=2, ensure_ascii=False))

source_counts

## 8. Retrieval Demo

This demo loads the FAISS retriever and runs sample searches. It is local and does not call Gemini.

In [ ]:
retriever = Retriever()
queries = [
    "What is retrieval augmented generation?",
    "What is MLOps?",
    "What is Kolmogorov-Arnold Network?",
]

rows = []
for query in queries:
    start = time.perf_counter()
    results = retriever.retrieve(query, top_k=3, use_reranking=False)
    elapsed = time.perf_counter() - start
    for result in results:
        rows.append({
            "query": query,
            "rank": result["rank"],
            "source": result["source"],
            "page": result["page"],
            "score": round(result["score"], 4),
            "latency_sec": round(elapsed, 4),
            "preview": result["chunk_text"][:180].replace("\n", " "),
        })

pd.DataFrame(rows)

## 9. RAG Question Answering Demo

The following cell is safe by default. It shows retrieved context for a QA prompt without calling Gemini. To run live generation, set `RUN_LLM_DEMOS = True` and ensure Gemini API quota is available.

In [ ]:
RUN_LLM_DEMOS = True
sample_question = "What is retrieval augmented generation?"

qa_retrieved = retriever.retrieve(sample_question, top_k=3, use_reranking=False)
print("Question:", sample_question)
print("Retrieved context preview:")
for item in qa_retrieved:
    print(f"- {item['source']} p.{item['page']}: {item['chunk_text'][:180].replace(chr(10), ' ')}")

if RUN_LLM_DEMOS:
    qa = QAGenerator(retriever=retriever)
    answer = qa.ask(sample_question, top_k=3)
    print(answer["answer"])
else:
    print("Live Gemini QA skipped. Set RUN_LLM_DEMOS=True to run this cell with API quota.")

## 10. Summary Demo

This cell avoids Gemini by default. It demonstrates the retrieved context that would ground the summary. Live summarization can be enabled with `RUN_LLM_DEMOS = True`.

In [ ]:
summary_topic = "Summarize retrieval augmented generation"
summary_context = retriever.retrieve(summary_topic, top_k=5, use_reranking=False)
print("Summary topic:", summary_topic)
print("Top retrieved sources:")
for item in summary_context[:3]:
    print(f"- {item['source']} p.{item['page']}")

if RUN_LLM_DEMOS:
    summarizer = Summarizer(retriever=retriever)
    summary = summarizer.summarize_topic(summary_topic, top_k=5)
    print(summary["summary"])
else:
    print("Live Gemini summary skipped. Set RUN_LLM_DEMOS=True to run this cell with API quota.")

## 11. MCQ Demo

This cell avoids Gemini by default. It shows how MCQ generation would be called. Live generation can be enabled with `RUN_LLM_DEMOS = True`.

In [ ]:
mcq_topic = "retrieval augmented generation"
print("MCQ topic:", mcq_topic)
print("Requested questions: 3")

if RUN_LLM_DEMOS:
    mcq_generator = MCQGenerator(retriever=retriever)
    mcq_result = mcq_generator.generate(mcq_topic, num_questions=3, top_k=5)
    print(f"Structured MCQs: {mcq_result['parsed_count']}/{mcq_result['requested_count']}")
    for i, mcq in enumerate(mcq_result["mcqs"], 1):
        print(f"Q{i}. {mcq['question']}  Answer: {mcq['answer']}")
else:
    print("Live Gemini MCQ generation skipped. Set RUN_LLM_DEMOS=True to run this cell with API quota.")

## 12. Evaluation Protocol

### Evaluation Split / Protocol

This project is a RAG system using pre-trained models, not a supervised fine-tuned classifier. Therefore, there is no train split for model fitting. The PDF corpus is used as the retrieval knowledge base.

Evaluation is done using a held-out set of manually revised questions. Development diagnostics were used to improve OCR, chunking, prompts, and display behavior. Final retrieval metrics are reported on the revised 15-question evaluation set.

Exact page-level matching is stricter than source-level retrieval because concepts may appear across adjacent pages or repeated slide headings. QA keyword scoring can also be strict because a correct generated answer may use different wording from the expected keywords.

In [ ]:
eval_path = EVALUATION_DIR / "eval_questions_revised.csv"
if eval_path.exists():
    eval_df = pd.read_csv(eval_path)
    print("Evaluation questions:", len(eval_df))
    display(eval_df.head())
else:
    eval_df = pd.DataFrame()
    print("Missing evaluation file:", eval_path)

## 13. Retrieval Evaluation Results

The notebook loads existing final evaluation outputs when available. If they are missing, it can run retrieval-only evaluation, which does not call Gemini.

In [ ]:
baseline_summary_path = EVALUATION_DIR / "eval_questions_revised_summary.json"
rerank_summary_path = EVALUATION_DIR / "eval_questions_revised_summary_rerank.json"

if baseline_summary_path.exists():
    baseline_summary = load_json_if_exists(baseline_summary_path)
else:
    baseline_summary = run_full_evaluation(eval_path=eval_path, mode="retrieval", top_k=5, use_reranking=False) if eval_path.exists() else None

if rerank_summary_path.exists():
    rerank_summary = load_json_if_exists(rerank_summary_path)
else:
    rerank_summary = run_full_evaluation(eval_path=eval_path, mode="retrieval", top_k=5, use_reranking=True, candidate_k=10) if eval_path.exists() else None

known_metrics = pd.DataFrame([
    {"setting": "Baseline", "Hit@3": 0.667, "Hit@5": 0.800, "MRR": 0.574},
    {"setting": "Lightweight reranking", "Hit@3": 0.733, "Hit@5": 0.867, "MRR": 0.578},
])

print("Final known metrics used in report:")
display(known_metrics)

print("Loaded baseline summary:")
print(json.dumps(baseline_summary, indent=2, ensure_ascii=False)[:2000] if baseline_summary else "Not available")

print("Loaded reranking summary:")
print(json.dumps(rerank_summary, indent=2, ensure_ascii=False)[:2000] if rerank_summary else "Not available")

## 14. Reranking Comparison

The lightweight reranker combines FAISS semantic score with keyword overlap. It improves the final revised evaluation results from 66.7% to 73.3% Hit@3 and from 80.0% to 86.7% Hit@5. The measured reranking overhead is negligible.

In [ ]:
comparison_path = EVALUATION_DIR / "reranking_comparison.json"
comparison = load_json_if_exists(comparison_path)

if comparison:
    print(json.dumps(comparison, indent=2, ensure_ascii=False)[:3000])
else:
    print("Known final reranking comparison:")
    print("Baseline Hit@3: 66.7%, Hit@5: 80.0%, MRR: 0.574")
    print("Reranking Hit@3: 73.3%, Hit@5: 86.7%, MRR: about 0.578")
    print("Source Hit@3 with reranking: 15/15")
    print("Source+Page Hit@3 with reranking: 11/15")

## 15. FastAPI and Streamlit Deployment Notes

The project includes both a Streamlit front-end and a FastAPI backend.

Run Streamlit:

```bash
streamlit run app.py
```

Run FastAPI:

```bash
uvicorn api:app --reload
```

FastAPI documentation:

```text
http://127.0.0.1:8000/docs
```

The API includes endpoints for health check, building the index, asking questions, summarization, and MCQ generation.

In [ ]:
# Lightweight FastAPI import validation. This does not start a server.
import api

print("FastAPI app title:", api.app.title)
print("Streamlit command: streamlit run app.py")
print("FastAPI command: uvicorn api:app --reload")
print("FastAPI docs: http://127.0.0.1:8000/docs")

## 16. Limitations

- The dataset is small: 6 PDFs.
- OCR quality depends on scan and slide quality.
- Exact page matching is stricter than source-level matching.
- Gemini generation depends on API availability and quota.
- Local fallback is intentionally explicit and slower.
- Lightweight reranking is heuristic rather than a trained reranker.
- QA keyword evaluation can penalize correct paraphrases.

## 17. Conclusion

The final project implements a complete RAG learning assistant over course PDFs. It satisfies the deployment requirement with Streamlit and FastAPI, uses a FAISS vector store over PDF chunks, relies on transformer-based embeddings and Gemini generation, and reports rigorous retrieval metrics on a revised 15-question evaluation set. The notebook is designed to support final reporting and presentation by loading existing artifacts and avoiding unnecessary OCR or generation calls by default.